# landcover_lca as goblin_lite uses it

goblin_lite (`impact_categories/climate_change.py`, class `ClimateChangeLandUse`) uses the **national** tree,
`landcover_lca.models` and `landcover_lca.lca_emission`. This notebook makes the same calls with the same arguments on
the test data in `goblin_lite_data/` (the national goblin_lite case pinned in `soc_test.py`), so it shows what
goblin_lite receives.

**Conventions (as in goblin_lite):**
- Every function returns **kg** of gas (or of N, P, PO4e); goblin_lite multiplies by `kg_to_kt = 1e-6`.
- The baseline dataset is keyed `-calibration_year` (here `-2020`) and reported under scenario index `-1`.
  Scenarios are keyed `0, 1, ...` and reported in the target year.
- Baseline: current and past land use are both the baseline data. Scenario: current land use is the scenario,
  past land use is the baseline; the transition matrix of the same key holds the areas that change.
- Every function used here takes `(land_use_data, past_land_use_data, transition_matrix, ef_country)`.
  `total_ch4_emission_cropland` and `total_n2o_emission_cropland` took `(ef_country, transition_matrix,
  land_use_data, past_land_use_data)` before; the old order now raises TypeError. goblin_lite still calls them in the old
  order, and in its scenarios passes the baseline as current land use: update both when it moves to this version.

Run from `tests/` (the data paths are relative).

In [ ]:
import pandas as pd
import landcover_lca.lca_emission as landuse_lca
import landcover_lca.models as landuse_models

ef_country = "ireland"
calibration_year = 2020
target_year = 2050
kg_to_kt = 1e-6
baseline_index = -1         # goblin_lite CommonParams.baseline_index
base = -calibration_year    # key of the baseline dataset

transition_data = pd.read_csv("./goblin_lite_data/transition_matrix.csv", index_col=0)
landuse_data = pd.read_csv("./goblin_lite_data/landuse_data_test_version.csv", index_col=0)

transition_matrix = landuse_models.load_transition_matrix(transition_data, ef_country, calibration_year, target_year)
land_use_data = landuse_models.load_land_use_data(landuse_data, calibration_year)

scenarios = [int(k) for k in land_use_data.keys() if int(k) >= 0]
print("datasets:", list(land_use_data.keys()), "scenarios:", scenarios)

## The functions goblin_lite calls
The three national totals and one total per land use and gas.

In [ ]:
# One function per land use and gas, and the totals: the calls goblin_lite makes.
FUNCTIONS = {
    ("cropland", "CO2"): landuse_lca.total_co2_emission_cropland,
    ("cropland", "CH4"): landuse_lca.total_ch4_emission_cropland,
    ("cropland", "N2O"): landuse_lca.total_n2o_emission_cropland,
    ("grassland", "CO2"): landuse_lca.total_co2_emission_grassland,
    ("grassland", "CH4"): landuse_lca.total_ch4_emission_grassland,
    ("grassland", "N2O"): landuse_lca.total_n2o_emission_grassland,
    ("forest", "CO2"): landuse_lca.total_co2_emission_forest,
    ("forest", "CH4"): landuse_lca.total_ch4_emission_forest,
    ("forest", "N2O"): landuse_lca.total_n2o_emission_forest,
    ("wetland", "CO2"): landuse_lca.total_co2_emission_wetland,
    ("wetland", "CH4"): landuse_lca.total_ch4_emission_wetland,
    ("wetland", "N2O"): landuse_lca.total_n2o_emission_wetland,
    ("total", "CO2"): landuse_lca.total_co2_emission,
    ("total", "CH4"): landuse_lca.total_ch4_emission,
    ("total", "N2O"): landuse_lca.total_n2o_emission,
}
LAND_USES = ["cropland", "grassland", "forest", "wetland", "total"]


def landuse_emissions(key, scenario_index, year):
    """kt of each gas by land use for one dataset (key), as ClimateChangeLandUse builds it."""
    df = pd.DataFrame(
        columns=["CO2", "CH4", "N2O"],
        index=pd.MultiIndex.from_product([[scenario_index], LAND_USES, [year]], names=["scenario", "land_use", "year"]),
        dtype=float,
    )
    for (land_use, gas), function in FUNCTIONS.items():
        kg = function(land_use_data[key], land_use_data[base], transition_matrix[key], ef_country)
        df.loc[(scenario_index, land_use, year), gas] = kg * kg_to_kt
    return df

## Baseline (`climate_change_landuse_past`)

In [ ]:
past = landuse_emissions(base, baseline_index, calibration_year)
past

## Scenarios (`climate_change_landuse_future`)

In [ ]:
future = pd.concat([landuse_emissions(sc, sc, target_year) for sc in scenarios])
future

## CO2e
goblin_lite takes the GWP values from `GoblinDataManager.get_AR_values()` (AR5 by default: CH4 28, N2O 265).

In [ ]:
AR_values = {"CH4": 28, "N2O": 265}  # AR5, as in goblin_lite's GoblinDataManager (illustration)

emissions = pd.concat([past, future])
emissions["CO2e"] = emissions["CO2"] + emissions["CH4"] * AR_values["CH4"] + emissions["N2O"] * AR_values["N2O"]
emissions

## Calls outside the land-use table

**Drained organic grassland N2O.** goblin_lite calls `drainage_n2O_organic_soils_in_grassland` in its agriculture
emissions (`soil_histosol_N_direct`), as the NID reports it in Agriculture 3.D. It is not part of
`total_n2o_emission_grassland`, so it is not counted twice.

In [ ]:
# N2O from drained organic grassland soils, which goblin_lite reports in agriculture as "soil_histosol_N_direct".
histosol_n2o_kt = {
    key: landuse_lca.drainage_n2O_organic_soils_in_grassland(
        land_use_data[key], land_use_data[base], transition_matrix[key], ef_country
    ) * kg_to_kt
    for key in [base] + scenarios
}
histosol_n2o_kt

**Horticultural peat.** Off-site CO2 from horticultural peat is already inside `total_co2_emission_wetland` (and so
`total_co2_emission`), for the year the land-use data carries (2020 for the baseline, 2050 for the scenarios; years
after the export data use the mean of the last three years). Do not add `horticulture_co2_peat_export` again. Its
share of wetland CO2:

In [ ]:
for key in [base] + scenarios:
    year = land_use_data[key].wetland.year
    peat_kt = landuse_lca.horticulture_co2_peat_export(ef_country, year) * kg_to_kt
    scenario_index = baseline_index if key == base else key
    wetland_kt = emissions.loc[(scenario_index, "wetland", year), "CO2"]
    print(f"dataset {key} ({year}): horticultural peat {peat_kt:,.1f} kt of wetland CO2 {wetland_kt:,.1f} kt")

**Forest carbon from CBM.** goblin_lite adds the CBM forest carbon (cbm_runner, `"Total Ecosystem"`, t C,
x `t_to_kt` x 3.67) to forest CO2 and to the total CO2. landcover_lca's forest CO2 covers drained and rewetted organic
soils and fire only; biomass, dead organic matter and mineral soil come from CBM. That step is not shown here.